# reasoning agent

In [1]:
import re, time, json, random, itertools, heapq, os
from collections import defaultdict

TEST_PATH = None          # csv/tsv/json/jsonl/parquet with the unseen queries
TEST_MODE = 'auto'        # 'auto' (picked per row), 'cwa' (negation as failure) or 'owa'
OUT_PATH = 'submission.csv'
REPORT_PATH = 'report.pdf'
README_PATH = 'README.txt'
MAX_EVAL = None           # cap rows per dataset for quick runs
FOL_TIMEOUT = 3.0         # seconds per resolution proof
SEED = 0
LOCAL_DIR = None          # offline option: folder with files named <dataset>_<split>.csv/jsonl/json/parquet

# hub ids, edit if loading fails
HF_IDS = {
    'ruletaker': ['tasksource/ruletaker', 'allenai/ruletaker'],
    'counterbench': ['CounterBench/CounterBench', 'counterbench/counterbench'],
    'folio': ['yale-nlp/FOLIO', 'tasksource/folio'],
}
HF_CONFIG = {}            # e.g. {'ruletaker': 'depth-3'}
MODE_OVERRIDE = {}        # e.g. {'ruletaker': 'owa'}
SPLIT_PREF = ['test', 'validation', 'dev', 'train']


## data

In [2]:
try:
    from datasets import load_dataset, get_dataset_config_names, Dataset, DatasetDict
except ImportError:
    load_dataset = None
    print('datasets missing, run: pip install datasets')

raw = {}


def read_table(path):
    import pandas as pd
    ext = os.path.splitext(path)[1].lower()
    if ext == '.csv': return pd.read_csv(path)
    if ext == '.tsv': return pd.read_csv(path, sep='\t')
    if ext == '.jsonl': return pd.read_json(path, lines=True)
    if ext == '.parquet': return pd.read_parquet(path)
    if ext == '.json':
        try:
            return pd.read_json(path)
        except Exception:
            with open(path, encoding='utf-8') as f:
                obj = json.load(f)
            if isinstance(obj, dict):
                obj = next((v for v in obj.values() if isinstance(v, list)), [obj])
            return pd.DataFrame(obj)
    raise ValueError('unknown file type ' + ext)


def to_dataset(df):
    # nested values (lists, dicts) are turned into json strings so arrow does not choke on mixed types
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].map(lambda v: v if isinstance(v, str) else json.dumps(v, ensure_ascii=False, default=str))
    return Dataset.from_pandas(df, preserve_index=False)


def load_raw_files(hid):
    # fallback when load_dataset fails (CounterBench did): read the repo files with pandas
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
    except Exception as e:
        print('  raw file fallback needs huggingface_hub:', e)
        return None
    try:
        files = list_repo_files(hid, repo_type='dataset')
    except Exception as e:
        print('  could not list files of', hid, ':', str(e).splitlines()[0][:150])
        return None
    out = {}
    for f in files:
        if os.path.splitext(f)[1].lower() not in ('.csv', '.tsv', '.json', '.jsonl', '.parquet'): continue
        try:
            df = read_table(hf_hub_download(hid, f, repo_type='dataset'))
            out[os.path.splitext(os.path.basename(f))[0]] = to_dataset(df)
            print('  read raw file', f, 'rows', len(df))
        except Exception as e:
            print('  skipped file', f, ':', str(e).splitlines()[0][:150])
    return DatasetDict(out) if out else None


def load_local(name):
    out = {}
    for f in sorted(os.listdir(LOCAL_DIR)):
        stem, ext = os.path.splitext(f)
        if stem.startswith(name + '_') and ext.lower() in ('.csv', '.tsv', '.json', '.jsonl', '.parquet'):
            out[stem[len(name) + 1:]] = to_dataset(read_table(os.path.join(LOCAL_DIR, f)))
    return DatasetDict(out) if out else None


def try_load(name):
    for hid in HF_IDS[name]:
        cfg = HF_CONFIG.get(name)
        try:
            ds = load_dataset(hid, cfg) if cfg else load_dataset(hid)
            return hid, cfg, ds
        except Exception as e:
            print(f'{name}: {hid} failed: {str(e).splitlines()[0][:200]}')
            if cfg is None:
                try:
                    cfgs = get_dataset_config_names(hid)
                except Exception:
                    cfgs = []
                if cfgs:
                    print(f'  configs for {hid}: {cfgs}')
                    try:
                        ds = load_dataset(hid, cfgs[0])
                        print(f'  using first config {cfgs[0]} (set HF_CONFIG to change)')
                        return hid, cfgs[0], ds
                    except Exception as e2:
                        print('  that failed too:', str(e2).splitlines()[0][:200])
            ds = load_raw_files(hid)
            if ds is not None:
                print(f'  loaded {hid} from its raw files')
                return hid, 'raw files', ds
    return None


def clip(v, n=1500):
    s = v if isinstance(v, str) else json.dumps(v, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + ' ...[cut]'


if load_dataset:
    for name in HF_IDS:
        got = None
        if LOCAL_DIR:
            ds = load_local(name)
            if ds is not None: got = (LOCAL_DIR, 'local', ds)
        if got is None and not LOCAL_DIR:
            got = try_load(name)
        if got is None:
            print(f'{name}: not loaded, no data for it')
            continue
        hid, cfg, ds = got
        raw[name] = ds
        print(f'\n{name} ({hid}, config {cfg})')
        for sp in ds:
            print(' ', sp, len(ds[sp]), ds[sp].column_names)
        sp0 = next(iter(ds))
        for k in range(min(2, len(ds[sp0]))):
            print(f' raw example {k} from {sp0}')
            for c, v in ds[sp0][k].items():
                print(f'   {c}: {clip(v)}')


c:\AI_Assignment\Assign_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



ruletaker (tasksource/ruletaker, config None)
  train 480152 ['context', 'question', 'label', 'config']
  test 151911 ['context', 'question', 'label', 'config']
  dev 75872 ['context', 'question', 'label', 'config']
 raw example 0 from train
   context: Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.
   question: Bob is kind.
   label: entailment
   config: depth-1
 raw example 1 from train
   context: Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.
   question: Dave is not smart.
   label: not entailment
   config: depth-1


Generating train split: 1200 examples [00:00, 3031.15 examples/s]


counterbench: CounterBench/CounterBench failed: An error occurred while generating the dataset
  configs for CounterBench/CounterBench: ['default']


Generating train split: 1200 examples [00:00, 7618.97 examples/s]


  that failed too: An error occurred while generating the dataset
  read raw file data_balanced_alpha_V1.json rows 1000
  read raw file data_balanced_backdoor_V2.json rows 200
  read raw file meta_model_alpha_V1.json rows 1000
  read raw file meta_model_backdoor_V2.json rows 200
  loaded CounterBench/CounterBench from its raw files

counterbench (CounterBench/CounterBench, config raw files)
  data_balanced_alpha_V1 1000 ['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
  data_balanced_backdoor_V2 200 ['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
  meta_model_alpha_V1 1000 ['model_id', 'background', 'variable_mapping', 'nonsense']
  meta_model_backdoor_V2 200 ['model_id', 'background', 'variable_mapping', 'nonsense']
 raw example 0 from data_balanced_alpha_V1
   question: Would Lumbo occur if not Ziklo instead of Ziklo?
   given_info: We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
   answer: no
   type

In [3]:
CTX = ['context', 'theory', 'premises', 'story', 'text', 'knowledge_base', 'kb', 'passage',
       'problem', 'prompt', 'input', 'description', 'scenario', 'background', 'given_info']
QRY = ['question', 'conclusion', 'statement', 'hypothesis', 'query', 'hypo']
LAB = ['label', 'answer', 'gold', 'target']
IDS = ['id', 'example_id', 'idx', 'qid', 'uid', 'question_id']

TRUE = {'true', 'entailment', 'entailed', 'yes', '1', 't'}
FALSE = {'false', 'not entailment', 'not_entailment', 'non-entailment', 'contradiction', 'no', '0', 'f'}
UNK = {'unknown', 'uncertain', 'neutral', 'na', 'n/a', 'not enough info', 'nei'}

def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]

def colmap(cols):
    m = {'ctx': pick(cols, CTX), 'q': pick(cols, QRY), 'lab': pick(cols, LAB), 'id': pick(cols, IDS)}
    m['pfol'] = next((c for c in cols if 'fol' in c.lower() and 'prem' in c.lower()), None)
    m['cfol'] = next((c for c in cols if 'fol' in c.lower() and 'concl' in c.lower()), None)
    return m

def norm_label(v, names=None):
    # unknown/uncertain count as not entailed, so they map to False
    if names is not None and isinstance(v, int):
        v = names[v]
    s = str(v).strip().lower()
    if s in TRUE: return True, s
    if s in FALSE: return False, s
    if s in UNK: return False, s
    return None, s

def txt(v):
    if v is None: return ''
    if isinstance(v, (list, tuple)): return '\n'.join(str(x) for x in v)
    return str(v)

def mkrow(r, m, names, i):
    g = r.get(m['lab']) if m['lab'] else None
    gold = rawl = None
    if g is not None and not (isinstance(g, float) and g != g):
        gold, rawl = norm_label(g, names)
    get = lambda k: txt(r.get(m[k])) if m[k] else ''
    return {'id': r.get(m['id'], i) if m['id'] else i, 'ctx': get('ctx'), 'q': get('q'),
            'gold': gold, 'raw': rawl, 'pfol': get('pfol'), 'cfol': get('cfol'), 'type': str(r.get('type', ''))}

def get_rows(sp, limit=None):
    m = colmap(sp.column_names)
    n = len(sp)
    idx = list(range(n))
    if limit and n > limit:
        idx = sorted(random.Random(SEED).sample(idx, limit))
    names = None
    if m['lab'] and m['lab'] in sp.features and hasattr(sp.features[m['lab']], 'names'):
        names = sp.features[m['lab']].names
    return [mkrow(sp[i], m, names, i) for i in idx], m

## horn parser

In [4]:
VARW = {'something', 'someone', 'everything', 'everyone', 'it', 'they', 'somebody', 'everybody', 'he', 'she'}
DET = ('the', 'a', 'an')

def stem(w):
    return w[:-1] if w.endswith('s') and not w.endswith('ss') else w

def ent_ok(tk):
    if not tk: return False
    if len(tk) == 1: return tk[0] not in DET and tk[0] not in ('not', 'is', 'are')
    return tk[0] == 'the' and len(tk) <= 3

def ent(tk):
    if len(tk) == 1 and tk[0] in VARW: return '?x'
    if tk[0] in DET: tk = tk[1:]
    return '_'.join(tk)

def clause(s):
    # returns (literals, frame) or None
    tk = s.split()
    for i in range(1, min(4, len(tk))):
        if not ent_ok(tk[:i]) or i >= len(tk): continue
        subj, rest = ent(tk[:i]), tk[i:]
        if rest[0] in ('is', 'are'):
            r, ng = rest[1:], False
            if r[:1] == ['not']: ng, r = True, r[1:]
            if r[:1] in (['a'], ['an']): r = r[1:]
            if not r or r[0] == 'the': continue
            return [(ng, '_'.join(r), (subj,))], ('u', subj)
        ng = False
        if rest[:2] in (['does', 'not'], ['do', 'not'], ['did', 'not']):
            ng, rest = True, rest[2:]
        elif rest[0] in ("doesn't", "don't", "didn't"):
            ng, rest = True, rest[1:]
        if len(rest) >= 2 and ent_ok(rest[1:]):
            v = stem(rest[0])
            return [(ng, v, (subj, ent(rest[1:])))], ('r', subj, v)
    return None

def clauses(text):
    # split on and/commas, glue bare adjectives or objects onto the previous clause
    pieces = [p.strip() for p in re.split(r'\s*,\s*and\s+|\s+and\s+|\s*,\s*', text) if p.strip()]
    out, frame = [], None
    for p in pieces:
        c = clause(p)
        if c is not None:
            out += c[0]; frame = c[1]
            continue
        if frame is None: return None
        tk = p.split()
        if frame[0] == 'u':
            ng = tk[:1] == ['not']
            if ng: tk = tk[1:]
            if not tk: return None
            out.append((ng, '_'.join(tk), (frame[1],)))
        else:
            if not ent_ok(tk): return None
            out.append((False, frame[2], (frame[1], ent(tk))))
    return out or None

def parse_rule(s):
    m = re.match(r'^if (.*?),? then (.*)$', s)
    if m:
        body, head = clauses(m.group(1)), clauses(m.group(2))
        if body is None or head is None: return None
        # a lone "it" in the head points back at the first body subject
        if any('?x' in l[2] for l in head) and not any('?x' in l[2] for l in body):
            sub = body[0][2][0]
            head = [(n, p, tuple(sub if a == '?x' else a for a in ar)) for n, p, ar in head]
        return [(body, h, s) for h in head]
    m = re.match(r'^(?:all |every |each )?(.*?)\b(?:things?|people|persons?|ones?)\s+(?:are|is)\s+(not\s+)?(.+)$', s)
    if m:
        pre = re.sub(r'^(?:all|every|each)\s*', '', m.group(1)).strip()
        body = []
        if pre:
            for a in re.split(r'\s*,\s*and\s+|\s+and\s+|\s*,\s*', pre):
                if not a.strip(): return None
                body.append((False, '_'.join(a.split()), ('?x',)))
        ng = bool(m.group(2))
        heads = [(ng, '_'.join(a.split()), ('?x',)) for a in re.split(r'\s*,\s*and\s+|\s+and\s+|\s*,\s*', m.group(3)) if a.strip()]
        if not heads: return None
        return [(body, h, s) for h in heads]
    m = re.match(r'^(?:everything|everyone|everybody)\s+(?:is|are)\s+(not\s+)?(.+)$', s)
    if m:
        ng = bool(m.group(1))
        return [([], (ng, '_'.join(a.split()), ('?x',)), s) for a in re.split(r'\s*,\s*and\s+|\s+and\s+|\s*,\s*', m.group(2)) if a.strip()]
    return None

def parse_ctx(text):
    facts, rules, bad = [], [], []
    for s in re.split(r'(?<=[.!?])\s+|\n+', text.strip()):
        s = re.sub(r'\s+', ' ', s.strip().rstrip('.!').strip().lower())
        if not s: continue
        if s.startswith('if ') or re.search(r'\b(things?|people|persons?|ones?)\s+(are|is)\b', s) \
                or re.match(r'^(everything|everyone|everybody)\b', s):
            r = parse_rule(s)
            if r is None: bad.append(s)
            else: rules += r
            continue
        c = clauses(s)
        if c is None or any('?x' in l[2] for l in c): bad.append(s)
        else: facts += c
    return facts, rules, bad

def parse_query(q):
    s = re.sub(r'\s+', ' ', q.strip().rstrip('.?! ').strip().lower())
    s = re.sub(r'^(q|query|question)\s*:\s*', '', s)
    return clauses(s)

## forward chaining

In [5]:
def new_store():
    return {'p': {}, 'n': {}, 'p0': {}, 'n0': {}}

def add_fact(st, neg, pred, tup):
    k = 'n' if neg else 'p'
    s = st[k].setdefault(pred, set())
    if tup in s: return False
    s.add(tup)
    if tup: st[k + '0'].setdefault((pred, tup[0]), []).append(tup)
    return True

def cand(st, pred, args, b):
    if args:
        a = args[0]
        if a[0] == '?': a = b.get(a)
        if a is not None: return st['p0'].get((pred, a), ())
    return st['p'].get(pred, ())

def match(args, tup, b):
    if len(args) != len(tup): return None
    b2 = b
    for a, v in zip(args, tup):
        if a[0] == '?':
            if a in b2:
                if b2[a] != v: return None
            else:
                if b2 is b: b2 = dict(b)
                b2[a] = v
        elif a != v:
            return None
    return b2

def prep(rule):
    body, head, text = rule
    left = [l for l in body if not l[0]]
    ng = [l for l in body if l[0]]
    ordd, bound = [], set()
    while left:
        left.sort(key=lambda l: -sum(1 for a in l[2] if a[0] != '?' or a in bound))
        l = left.pop(0)
        ordd.append(l)
        bound |= {a for a in l[2] if a[0] == '?'}
    return (ordd, ng, head, text, {l[1] for l in body})

def fire(r, st, dom, mode, deriv):
    pos, ng, head, text, _ = r
    hn, hp, ha = head
    added = []

    def done(b, prem):
        fv = []
        for l in ng + [head]:
            for a in l[2]:
                if a[0] == '?' and a not in b and a not in fv: fv.append(a)
        for combo in itertools.product(dom, repeat=len(fv)):
            bb = dict(b)
            bb.update(zip(fv, combo))
            pr = list(prem)
            ok = True
            for (_, p, a) in ng:
                t = tuple(bb.get(x, x) for x in a)
                if mode == 'owa':
                    ok = t in st['n'].get(p, ())
                    if ok: pr.append(('f', (True, p, t)))
                else:
                    ok = t not in st['p'].get(p, ())
                    if ok: pr.append(('naf', (False, p, t)))
                if not ok: break
            if not ok: continue
            ht = tuple(bb.get(x, x) for x in ha)
            if add_fact(st, hn, hp, ht):
                deriv[(hn, hp, ht)] = (text, pr)
                added.append(hp)

    def rec(k, b, prem):
        if k == len(pos):
            done(b, prem)
            return
        _, p, a = pos[k]
        for t in list(cand(st, p, a, b)):
            b2 = match(a, t, b)
            if b2 is not None:
                rec(k + 1, b2, prem + [('f', (False, p, t))])

    rec(0, {}, [])
    return added

def strata(rules, mode):
    if mode == 'owa' or not any(l[0] for r in rules for l in r[0]):
        return [rules]
    lv = defaultdict(int)
    for _ in range(len(rules) + 3):
        ch = False
        for body, head, _ in rules:
            for (n, p, _a) in body:
                need = lv[p] + (1 if n else 0)
                if lv[head[1]] < need:
                    lv[head[1]] = need
                    ch = True
        if not ch: break
    else:
        return [rules]
    g = defaultdict(list)
    for r in rules: g[lv[r[1][1]]].append(r)
    return [g[k] for k in sorted(g)]

def closure(facts, rules, mode):
    st, given, deriv = new_store(), set(), {}
    dom = set()
    for f in facts:
        add_fact(st, *f)
        given.add(f)
        dom |= set(f[2])
    dropped = 0
    keep = []
    for r in rules:
        if mode == 'cwa' and r[1][0]:
            dropped += 1
            continue
        keep.append(r)
        for l in r[0] + [r[1]]:
            dom |= {a for a in l[2] if a[0] != '?'}
    for grp in strata(keep, mode):
        prs = [prep(r) for r in grp]
        changed = None
        while True:
            chp = set()
            for r in prs:
                if changed is not None and not (r[4] & changed): continue
                chp |= set(fire(r, st, dom, mode, deriv))
            if not chp: break
            changed = chp
    return st, deriv, given, dropped

def holds(st, lit, mode):
    n, p, a = lit
    if any(x[0] == '?' for x in a):
        if n: return False
        return any(match(a, t, {}) is not None for t in st['p'].get(p, ()))
    if not n: return a in st['p'].get(p, ())
    if mode == 'owa': return a in st['n'].get(p, ())
    return a not in st['p'].get(p, ())

def fmt(l):
    n, p, a = l
    return ('not ' if n else '') + p + '(' + ', '.join(a) + ')'

def explain(f, deriv, given, lines, d, seen):
    pad = '  ' * d
    if f in seen:
        lines.append(pad + fmt(f) + '   [shown above]')
        return
    seen.add(f)
    if f in deriv:
        text, prem = deriv[f]
        lines.append(pad + fmt(f) + '   [by: ' + text + ']')
        for kind, pr in prem:
            if kind == 'naf': lines.append(pad + '  ' + fmt((True, pr[1], pr[2])) + '   [no proof of ' + fmt(pr) + ']')
            else: explain(pr, deriv, given, lines, d + 1, seen)
    else:
        lines.append(pad + fmt(f) + ('   [given]' if f in given else '   [not derived]'))

## fol resolution

In [6]:
def fnorm(s):
    for a, b in (('<->', '↔'), ('->', '→'), ('&', '∧'), ('~', '¬')):
        s = s.replace(a, b)
    return s

FT = re.compile(r'\s*(∀|∃|¬|∧|∨|⊕|→|↔|⟷|⇔|⇒|≠|=|\(|\)|,|\w+)')

def ftok(s):
    s = fnorm(s).strip().rstrip('.')
    out, i = [], 0
    while i < len(s):
        m = FT.match(s, i)
        if not m: raise ValueError('bad char near ' + repr(s[i:i + 12]))
        out.append(m.group(1))
        i = m.end()
    return out

def fparse(s):
    t = ftok(s)
    i = [0]
    bound = []

    def peek():
        return t[i[0]] if i[0] < len(t) else None

    def eat(x=None):
        v = peek()
        if v is None or (x is not None and v != x): raise ValueError(f'expected {x} got {v}')
        i[0] += 1
        return v

    def term():
        n = eat().lower()
        if not re.fullmatch(r'\w+', n): raise ValueError('bad term ' + n)
        if peek() == '(':
            eat('(')
            a = [term()]
            while peek() == ',':
                eat(',')
                a.append(term())
            eat(')')
            return (n,) + tuple(a)
        return '?' + n if n in bound else n

    def atom():
        a = term()
        if peek() in ('=', '≠'):
            op = eat()
            b = term()
            f = ('atom', '=', (a, b))
            return f if op == '=' else ('not', f)
        if isinstance(a, tuple): return ('atom', a[0], a[1:])
        return ('atom', a, ())

    def un():
        v = peek()
        if v == '¬':
            eat()
            return ('not', un())
        if v in ('∀', '∃'):
            eat()
            vs = []
            while True:
                vs.append(eat().lower())
                if peek() == ',':
                    eat(',')
                    continue
                nx = peek()
                if nx is not None and re.fullmatch(r'\w+', nx) and (i[0] + 1 >= len(t) or t[i[0] + 1] != '('):
                    continue
                break
            bound.extend(vs)
            f = un()
            del bound[-len(vs):]
            for n in reversed(vs): f = ('all' if v == '∀' else 'ex', n, f)
            return f
        if v == '(':
            eat('(')
            f = iff()
            eat(')')
            return f
        return atom()

    def and_():
        f = un()
        while peek() == '∧':
            eat()
            f = ('and', f, un())
        return f

    def or_():
        f = and_()
        while peek() in ('∨', '⊕'):
            op = eat()
            f = ('or' if op == '∨' else 'xor', f, and_())
        return f

    def imp():
        f = or_()
        if peek() in ('→', '⇒'):
            eat()
            return ('imp', f, imp())
        return f

    def iff():
        f = imp()
        while peek() in ('↔', '⟷', '⇔'):
            eat()
            f = ('iff', f, imp())
        return f

    f = iff()
    if i[0] != len(t): raise ValueError('trailing tokens')
    return f

def nnf(f, neg=False):
    op = f[0]
    if op == 'atom': return ('not', f) if neg else f
    if op == 'not': return nnf(f[1], not neg)
    if op in ('and', 'or'):
        o = op if not neg else ('or' if op == 'and' else 'and')
        return (o, nnf(f[1], neg), nnf(f[2], neg))
    if op == 'imp': return nnf(('or', ('not', f[1]), f[2]), neg)
    if op == 'iff': return nnf(('and', ('imp', f[1], f[2]), ('imp', f[2], f[1])), neg)
    if op == 'xor': return nnf(('or', ('and', f[1], ('not', f[2])), ('and', ('not', f[1]), f[2])), neg)
    if op in ('all', 'ex'):
        o = op if not neg else ('ex' if op == 'all' else 'all')
        return (o, f[1], nnf(f[2], neg))
    raise ValueError(op)

cnt = [0]

def sub(t, env):
    if isinstance(t, str): return env.get(t, t)
    return (t[0],) + tuple(sub(x, env) for x in t[1:])

def skolem(f, env, univ):
    op = f[0]
    if op == 'atom': return ('atom', f[1], tuple(sub(a, env) for a in f[2]))
    if op == 'not': return ('not', skolem(f[1], env, univ))
    if op in ('and', 'or'): return (op, skolem(f[1], env, univ), skolem(f[2], env, univ))
    cnt[0] += 1
    e = dict(env)
    if op == 'all':
        v = '?#%d' % cnt[0]
        e['?' + f[1]] = v
        return skolem(f[2], e, univ + [v])
    nm = 'sk#%d' % cnt[0]
    e['?' + f[1]] = (nm,) + tuple(univ) if univ else nm
    return skolem(f[2], e, univ)

def to_cnf(f, cap=2000):
    op = f[0]
    if op == 'and': return to_cnf(f[1], cap) + to_cnf(f[2], cap)
    if op == 'or':
        a, b = to_cnf(f[1], cap), to_cnf(f[2], cap)
        if len(a) * len(b) > cap: raise ValueError('cnf too big')
        return [x + y for x in a for y in b]
    if op == 'not': return [[(True, f[1][1], f[1][2])]]
    return [[(False, f[1], f[2])]]

def clauses_of(f):
    out = []
    for c in to_cnf(skolem(nnf(f), {}, [])):
        c = list(dict.fromkeys(c))
        if any((not l[0], l[1], l[2]) in c for l in c): continue
        out.append(tuple(c))
    return out

def walk(t, s):
    while isinstance(t, str) and t[0] == '?' and t in s: t = s[t]
    return t

def occurs(v, t, s):
    t = walk(t, s)
    if t == v: return True
    return isinstance(t, tuple) and any(occurs(v, x, s) for x in t[1:])

def unify(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b: return s
    if isinstance(a, str) and a[0] == '?':
        if occurs(a, b, s): return None
        return {**s, a: b}
    if isinstance(b, str) and b[0] == '?':
        if occurs(b, a, s): return None
        return {**s, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and len(a) == len(b) and a[0] == b[0]:
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None: return None
        return s
    return None

def subst(t, s):
    if isinstance(t, str):
        return subst(s[t], s) if t[0] == '?' and t in s else t
    return (t[0],) + tuple(subst(x, s) for x in t[1:])

def ren(c, tag):
    def r(t):
        if isinstance(t, str): return t + tag if t[0] == '?' else t
        return (t[0],) + tuple(r(x) for x in t[1:])
    return tuple((n, p, tuple(r(a) for a in ar)) for n, p, ar in c)

def depth(t):
    return 0 if isinstance(t, str) else 1 + max([depth(x) for x in t[1:]] + [0])

def ckey(c):
    m = {}
    def r(t):
        if isinstance(t, str): return m.setdefault(t, '?%d' % len(m)) if t[0] == '?' else t
        return (t[0],) + tuple(r(x) for x in t[1:])
    ls = sorted(c, key=lambda l: (l[1], l[0]))
    return tuple((n, p, tuple(r(a) for a in ar)) for n, p, ar in ls)

def tstr(t):
    if isinstance(t, str): return t.replace('?#', 'V')
    return t[0] + '(' + ','.join(tstr(x) for x in t[1:]) + ')'

def cstr(c):
    if not c: return '[]'
    return ' | '.join(('~' if n else '') + p + ('(' + ','.join(tstr(a) for a in ar) + ')' if ar else '') for n, p, ar in c)

def relevant(axioms, goals):
    ps = {l[1] for c in goals for l in c}
    keep, rest = [], list(axioms)
    while True:
        add = [c for c in rest if any(l[1] in ps for l in c)]
        if not add: break
        for c in add:
            keep.append(c)
            ps |= {l[1] for l in c}
        rest = [c for c in rest if c not in add]
    return keep

def refute(axioms, goals, tmax, maxlit=12, maxdepth=4, maxn=40000):
    t0 = time.perf_counter()
    cl, par, seen = {}, {}, set()
    by = defaultdict(list)
    sos = []
    nid = [0]

    def new(c, parents, why):
        k = ckey(c)
        if k in seen: return None
        seen.add(k)
        nid[0] += 1
        cl[nid[0]] = c
        par[nid[0]] = (parents, why)
        return nid[0]

    def usable(cid):
        for j, (n, p, a) in enumerate(cl[cid]): by[(n, p)].append((cid, j))

    for c in axioms:
        k = new(c, (), 'premise')
        if k: usable(k)
    for c in goals:
        k = new(c, (), 'negated conclusion')
        if k: heapq.heappush(sos, (len(c), k))

    def found(k):
        keep, stack = set(), [k]
        while stack:
            x = stack.pop()
            if x in keep: continue
            keep.add(x)
            stack += list(par[x][0])
        lines = []
        for x in sorted(keep):
            ps, why = par[x]
            lines.append(f'{x}: {cstr(cl[x])}   [{why}' + (' ' + ','.join(map(str, ps)) if ps else '') + ']')
        return '\n'.join(lines)

    def push(c, parents, why):
        if len(c) > maxlit: return None
        if any(depth(a) > maxdepth for _, _, ar in c for a in ar): return None
        c = tuple(dict.fromkeys(c))
        if any((not l[0], l[1], l[2]) in c for l in c): return None
        k = new(c, parents, why)
        if k is None: return None
        if not c: return k
        heapq.heappush(sos, (len(c), k))
        return k

    while sos:
        if time.perf_counter() - t0 > tmax: return False, 'timeout', ''
        if nid[0] > maxn: return False, 'limit', ''
        _, g = heapq.heappop(sos)
        gc = cl[g]
        usable(g)
        for a in range(len(gc)):
            for b in range(a + 1, len(gc)):
                if gc[a][:2] != gc[b][:2]: continue
                s = {}
                for x, y in zip(gc[a][2], gc[b][2]):
                    s = unify(x, y, s)
                    if s is None: break
                if s is None: continue
                f = tuple((n, p, tuple(subst(x, s) for x in ar)) for k2, (n, p, ar) in enumerate(gc) if k2 != b)
                k = push(f, (g,), 'factor')
                if k and not f: return True, 'proved', found(k)
        for i, (n, p, a) in enumerate(gc):
            for (oid, j) in list(by.get((not n, p), ())):
                oc = ren(cl[oid], '_%d' % nid[0])
                oa = oc[j][2]
                if len(oa) != len(a): continue
                s = {}
                for x, y in zip(a, oa):
                    s = unify(x, y, s)
                    if s is None: break
                if s is None: continue
                rest = [l for k2, l in enumerate(gc) if k2 != i] + [l for k2, l in enumerate(oc) if k2 != j]
                f = tuple((n2, p2, tuple(subst(x, s) for x in ar)) for n2, p2, ar in rest)
                k = push(f, (g, oid), 'resolve')
                if k and not f: return True, 'proved', found(k)
        if time.perf_counter() - t0 > tmax: return False, 'timeout', ''
    return False, 'saturated', ''

def depth_at(t, i):
    d = 0
    for ch in t[:i]:
        if ch == '(': d += 1
        elif ch == ')': d -= 1
    return d


def fparse_fixed(s):
    # FOLIO has some formulas with unbalanced brackets. Try the plain parse first,
    # then a few repairs depending on whether there are too many or too few ")".
    try:
        return fparse(s)
    except Exception as err:
        first = err
    t = fnorm(s).strip().rstrip('.')
    n = t.count('(') - t.count(')')
    cands = []
    if n < 0:
        k = -n
        a = t
        for _ in range(k):
            if a.rstrip().endswith(')'): a = a.rstrip()[:-1]
        cands.append(a)
        m = re.match(r'^((?:[∀∃]\s*\w+\s*)+)(.*)$', t)
        if m: cands.append(m.group(1) + '(' * k + m.group(2))
        cands.append('(' * k + t)
    elif n > 0:
        k = n
        # a missing ")" is usually just before an arrow that sits too deep
        for i, ch in enumerate(t):
            if ch in '→↔' and depth_at(t, i) >= 2:
                cands.append(t[:i].rstrip() + ')' * k + ' ' + t[i:])
        cands.append(t + ')' * k)
    for c in cands:
        try:
            f = fparse(c)
        except Exception:
            continue
        # if the formula starts with a quantifier its scope must cover everything
        if re.match(r'^\s*[∀∃]', c) and f[0] not in ('all', 'ex'): continue
        return f
    raise first


def fol_predict(pfol, cfol, stats):
    t0 = time.perf_counter()
    prem = [x for x in re.split(r'\n+', pfol) if x.strip()]
    axioms = []
    for s in prem:
        try:
            axioms += clauses_of(fparse_fixed(s))
        except Exception as e:
            stats['fol_bad'] += 1
            if len(stats['bad_samples']) < 10: stats['bad_samples'].append('FOL: ' + s[:120] + ' (' + str(e)[:40] + ')')
    try:
        goals = clauses_of(('not', fparse_fixed(cfol)))
    except Exception as e:
        stats['fol_bad_concl'] += 1
        if len(stats['bad_samples']) < 10: stats['bad_samples'].append('FOL concl: ' + cfol[:120])
        return False, 'bad conclusion', ''
    t1 = time.perf_counter()
    ax = relevant(axioms, goals)
    ok, why, proof = refute(ax, goals, FOL_TIMEOUT)
    stats['tp'] += t1 - t0
    stats['tr'] += time.perf_counter() - t1
    stats['fol_' + why] += 1
    return ok, why, proof

## causal models (CounterBench)

CounterBench gives a small causal graph in words and asks a counterfactual question. Each variable is true or false, 
an effect is a rule over its causes (and, or, not), the roots are unknown unless observed. 
I list every assignment of the roots that fits the evidence, force the variables named in the question (do), 
evaluate the model by backward chaining and answer yes when most of the weight says yes.


In [7]:
CF_PICK = {'instead': 'I1', 'plain': 'P1', 'nested': 'N1'}
CF_VARIANTS = {'instead': ['I1', 'I2'], 'plain': ['P1', 'P2'], 'nested': ['N1', 'N2', 'N3']}
LIT = re.compile(r'^(not\s+)?([A-Za-z_]\w*)$', re.I)


def cf_lits(text):
    # "not A and B" -> [(True, 'A'), (False, 'B')]
    out = []
    for p in re.split(r'\s+and\s+|\s*,\s*', text.strip().rstrip('.')):
        p = p.strip()
        if not p: continue
        m = LIT.match(p)
        if not m: return None
        out.append((bool(m.group(1)), m.group(2)))
    return out or None


def parse_cause_expr(t):
    # a cause is an or of and groups, e.g. "A and B together" or "A or not B"
    out = []
    for term in re.split(r'\s+or\s+', t.strip()):
        ls = cf_lits(term)
        if ls is None: return None
        out.append(ls)
    return out


def is_causal(ctx):
    c = ctx.lower()
    return 'we know that' in c and re.search(r'\bcauses?\b', c) is not None


def parse_causal(ctx):
    parts = [p.strip().rstrip('.').strip() for p in re.split(r'\.\s+', ctx.strip())]
    know, obs, prior = '', [], {}
    for p in parts:
        for nm, pr in re.findall(r'(\w+)\s*~\s*bern\(\s*([\d.]+)\s*\)', p, flags=re.I):
            prior[nm] = float(pr)
        low = p.lower()
        if low.startswith('we know that'):
            know = p[len('we know that'):].strip()
        elif low.startswith('we observed'):
            o = cf_lits(p[len('we observed'):])
            if o is None: return None
            obs += o
    if not know: return None
    know = re.sub(r',\s*and\s+', ', ', know)
    pieces = [re.sub(r'^and\s+', '', p.strip()) for p in re.split(r',\s*', know) if p.strip()]
    cl, prev = [], None
    for p in pieces:
        m = re.match(r'^(.+?)\s+causes?\s+(.+)$', p)
        if m:
            ce = parse_cause_expr(re.sub(r'\s+together$', '', m.group(1).strip()))
            effs = cf_lits(m.group(2))
            if ce is None or effs is None: return None
            prev = [ce, list(effs)]
            cl.append(prev)
        else:
            # no verb: more effects of the previous cause ("A causes B, C and D")
            effs = cf_lits(p)
            if prev is None or effs is None: return None
            prev[1].extend(effs)
    eqs, names = defaultdict(list), set(prior) | {v for _, v in obs}
    for ce, effs in cl:
        for ne, ev in effs:
            eqs[ev].append((ce, ne))
            names.add(ev)
            for term in ce:
                names |= {cv for _, cv in term}
    roots = sorted(n for n in names if n not in eqs)
    return {'eqs': dict(eqs), 'roots': roots, 'obs': obs, 'prior': prior}


def cf_value(model, assign, do):
    # evaluates every variable of the model by backward lookup (a variable that is set by do ignores its own causes)
    state = {}

    def get(v, seen=()):
        if v in do: return do[v]
        if v in assign: return assign[v]
        if v in state: return state[v]
        if v not in model['eqs'] or v in seen: return False
        res = False
        for ce, ne in model['eqs'][v]:
            on = any(all(get(cv, seen + (v,)) != cn for cn, cv in term) for term in ce)
            res = res or (on != ne)
        state[v] = res
        return res
    return get


def cf_worlds(model):
    R = model['roots'][:14]
    for bits in itertools.product([False, True], repeat=len(R)):
        a = dict(zip(R, bits))
        w = 1.0
        for k, b in a.items():
            p = model['prior'].get(k, 0.5)
            w *= p if b else 1 - p
        yield a, w


def qlit(text):
    t = text.strip()
    neg = False
    if t.lower().endswith(' not'): t, neg = t[:-4].strip(), True
    ls = cf_lits(t)
    if ls is None or len(ls) != 1: return None
    return (ls[0][0] != neg, ls[0][1])


def parse_cf_question(q):
    s = re.sub(r'\s+', ' ', q.strip())
    m = re.match(r'(?i)^assume\s+(.+?)\s*,?\s+and based on this assumption,?\s+(?:further\s+)?suppose\s+(.+?)[.,]?\s+would\s+(.+?)\s+occur\s*\??$', s)
    if m:
        a, b, y = cf_lits(m.group(1)), cf_lits(m.group(2)), qlit(m.group(3))
        return ('nested', y, [a, b], None) if a and b and y else None
    m = re.match(r'(?i)^would\s+(.+?)\s+occur\s+if\s+(.+?)\s+instead of\s+(.+?)\s*\??$', s)
    if m:
        y, c, f = qlit(m.group(1)), cf_lits(m.group(2)), cf_lits(m.group(3))
        return ('instead', y, [c], f) if y and c and f else None
    m = re.match(r'(?i)^would\s+(.+?)\s+occur\s+if\s+(.+?)\s*\??$', s)
    if m:
        y, c = qlit(m.group(1)), cf_lits(m.group(2))
        return ('plain', y, [c], None) if y and c else None
    return None


def causal_run(ctx, q, pick=None):
    pick = pick or CF_PICK
    model, pq = parse_causal(ctx), parse_cf_question(q)
    if model is None or pq is None: return None, None
    form, y, steps, fact = pq
    var = pick[form]
    obs = list(model['obs'])
    if form == 'instead' and var == 'I1': obs += fact       # "instead of A" says A was what really happened
    if form == 'plain' and var == 'P2': obs = []
    if form == 'nested' and var == 'N2': steps = steps[:1]
    if form == 'nested' and var == 'N3': steps = steps[-1:]
    do = {}
    for g in steps:
        for n, v in g: do[v] = not n
    worlds = list(cf_worlds(model))
    ok = [(a, w) for a, w in worlds if all(cf_value(model, a, {})(v) != n for n, v in obs)]
    if not ok or sum(w for _, w in ok) <= 0: ok = worlds
    yes = tot = 0.0
    res = []
    for a, w in ok:
        t = cf_value(model, a, do)(y[1]) != y[0]
        res.append((a, w, t))
        tot += w
        yes += w * t
    return yes / tot > 0.5, {'model': model, 'form': form, 'variant': var, 'do': do, 'obs': obs, 'query': y, 'worlds': res}


def causal_answer(ctx, q, pick=None):
    return causal_run(ctx, q, pick)[0]


def causal_predict(ctx, q, stats=None):
    a = causal_answer(ctx, q)
    if a is None:
        if stats is not None: stats['bad_causal'] += 1
        return False
    return a


def lit_s(n, v): return ('not ' if n else '') + v


def causal_trace(ctx, q):
    ans, info = causal_run(ctx, q)
    if info is None: return 'not parsed'
    L = ['model (each variable and what sets it):']
    for v, lst in info['model']['eqs'].items():
        for ce, ne in lst:
            expr = ' or '.join('(' + ' and '.join(lit_s(n, c) for n, c in t) + ')' for t in ce)
            L.append(f'  {v} = ' + ('not ' if ne else '') + expr)
    L.append('roots: ' + ', '.join(info['model']['roots']))
    L.append('evidence: ' + (', '.join(lit_s(n, v) for n, v in info['obs']) or 'none'))
    L.append(f"question form {info['form']}, reading {info['variant']}, intervention " +
             ', '.join(f'{v}={int(b)}' for v, b in info['do'].items()))
    L.append(f"worlds that fit the evidence: {len(info['worlds'])}")
    for a, w, t in info['worlds'][:6]:
        L.append('  roots ' + ', '.join(f'{k}={int(b)}' for k, b in a.items()) + f'  weight {w:.2f}  ->  {lit_s(*info["query"])} is {t}')
    L.append(f'answer: {ans}')
    return '\n'.join(L)


def tune_causal(rows):
    # picks, per question form, the reading of the question that agrees most with the public labels
    by = defaultdict(list)
    for r in rows:
        if r['gold'] is None or not is_causal(r['ctx']): continue
        pq = parse_cf_question(r['q'])
        if pq: by[pq[0]].append(r)
    for f, rs in by.items():
        best = None
        print(f'  causal form {f}: {len(rs)} rows')
        for v in CF_VARIANTS[f]:
            pick = dict(CF_PICK)
            pick[f] = v
            c = sum((causal_answer(r['ctx'], r['q'], pick) or False) == r['gold'] for r in rs)
            print(f'    reading {v}: {100.0 * c / len(rs):.2f}%')
            if best is None or c > best[1]: best = (v, c)
        CF_PICK[f] = best[0]
    print('  picked readings:', CF_PICK)
    ty = defaultdict(lambda: [0, 0])
    for r in rows:
        if r['gold'] is None or not is_causal(r['ctx']): continue
        p = causal_answer(r['ctx'], r['q'])
        ty[r['type']][0] += (bool(p) == r['gold'])
        ty[r['type']][1] += 1
    for t, (c, n) in sorted(ty.items()):
        print(f'  question type {t or "unknown"}: {100.0 * c / n:.2f}% ({c}/{n})')


## predict

In [8]:
cache = {}


def new_stats():
    s = defaultdict(float)
    s['bad_samples'] = []
    return s


def horn_predict(ctx, q, mode, stats):
    key = (ctx, mode)
    e = cache.get(key)
    if e is None:
        t0 = time.perf_counter()
        facts, rules, bad = parse_ctx(ctx)
        used = mode
        if mode == 'auto':
            # clean rule-style text gets negation as failure, messy text (FOLIO style) gets open world
            tot = len(facts) + len(rules) + len(bad)
            used = 'cwa' if tot and len(bad) <= 0.2 * tot else 'owa'
        t1 = time.perf_counter()
        st, deriv, given, dropped = closure(facts, rules, used)
        stats['tp'] += t1 - t0
        stats['tr'] += time.perf_counter() - t1
        stats['bad_sent'] += len(bad)
        stats['ok_sent'] += len(facts) + len(rules)
        stats['neg_head_dropped'] += dropped
        for b in bad:
            if len(stats['bad_samples']) < 10 and b not in stats['bad_samples']: stats['bad_samples'].append(b)
        if len(cache) > 3000: cache.clear()
        e = cache[key] = (st, deriv, given, used)
    t0 = time.perf_counter()
    ql = parse_query(q)
    if ql is None:
        stats['bad_query'] += 1
        if len(stats['bad_samples']) < 10: stats['bad_samples'].append('QUERY: ' + q[:120])
        stats['tp'] += time.perf_counter() - t0
        return False
    ans = all(holds(e[0], l, e[3]) for l in ql)
    stats['tr'] += time.perf_counter() - t0
    return ans


def predict(r, mode, stats):
    # if the row comes with FOL strings use resolution, otherwise parse the English as horn rules
    if r['pfol'].strip() and r['cfol'].strip():
        stats['n_fol'] += 1
        ok, why, _ = fol_predict(r['pfol'], r['cfol'], stats)
        if why != 'bad conclusion':
            return ok
        stats['fol_to_nl'] += 1
    if is_causal(r['ctx']):
        stats['n_causal'] += 1
        return causal_predict(r['ctx'], r['q'], stats)
    return horn_predict(r['ctx'], r['q'], mode, stats)


## evaluation on public data

In [9]:
def evaluate(name, rows, mode):
    stats = new_stats()
    n = c = 0
    tp = tn = fp = fn = 0
    wrong, preds = [], []
    t0 = time.perf_counter()
    for r in rows:
        if r['gold'] is None: continue
        p = predict(r, mode, stats)
        preds.append(p)
        n += 1
        c += (p == r['gold'])
        if p == r['gold']: tp += p; tn += (not p)
        else:
            fp += p; fn += (not p)
            if len(wrong) < 3: wrong.append((r, p))
    tt = time.perf_counter() - t0
    res = {'n': n, 'correct': c, 'acc': 100.0 * c / max(n, 1), 'time': tt, 'tp': stats['tp'], 'tr': stats['tr'],
           'mode': mode, 'bad_sent': stats['bad_sent'], 'ok_sent': stats['ok_sent'], 'bad_query': stats['bad_query'],
           'fol_bad': stats['fol_bad'], 'n_fol': stats['n_fol'], 'conf': (tp, tn, fp, fn), 'preds': preds}
    for k in list(stats):
        if k.startswith('fol_') and k not in res: res[k] = stats[k]
    return res, stats, wrong


def nl_only(rows):
    return [dict(r, pfol='', cfol='') for r in rows]


results, keep_rows, wrong_ex, bad_ex = {}, {}, {}, {}


def run_one(key, rows, mode):
    print('  mode:', mode)
    res, stats, wrong = evaluate(key, rows, mode)
    results[key] = res
    wrong_ex[key] = wrong
    bad_ex[key] = stats['bad_samples']
    keep_rows[key] = (rows, mode)
    print(f"  accuracy {res['acc']:.2f}% ({res['correct']}/{res['n']}), time {res['time']:.2f}s "
          f"(parse {res['tp']:.2f}s, reason {res['tr']:.2f}s)")
    print(f"  confusion tp,tn,fp,fn = {res['conf']}")
    if stats['n_causal']: print(f"  causal rows {int(stats['n_causal'])}, not parsed {int(stats['bad_causal'])}")
    print(f"  unparsed context sentences {int(res['bad_sent'])} of {int(res['bad_sent'] + res['ok_sent'])}, bad queries {int(res['bad_query'])}, "
          f"fol rows {int(res['n_fol'])}, fol parse failures {int(res['fol_bad'])}")
    extra = {k: int(v) for k, v in res.items() if k.startswith('fol_') and k not in ('fol_bad', 'n_fol')}
    if extra: print('  fol outcomes:', extra)
    for s in bad_ex[key]: print('   parse fail:', s)


for name, ds in raw.items():
    if name == 'counterbench':
        # several files, only the ones with a question and an answer are data (the meta_model files are not)
        rows, used = [], []
        for spn in ds:
            mm = colmap(ds[spn].column_names)
            if not mm['lab'] or not mm['q']: continue
            rr, m = get_rows(ds[spn], MAX_EVAL)
            for r in rr: r['id'] = f'{spn}-{r["id"]}'
            rows += rr
            used.append(f'{spn} {len(rr)}')
        sp, n_avail = 'files ' + ', '.join(used), len(rows)
    else:
        sp = next((s for s in SPLIT_PREF if s in ds), None)
        if sp is None: sp = next(iter(ds))
        rows, m = get_rows(ds[sp], MAX_EVAL)
        n_avail = len(ds[sp])
    lab = [r for r in rows if r['gold'] is not None]
    print(f'\n{name}: split {sp}, {n_avail} rows available, {len(rows)} used, {len(lab)} labelled')
    print('  column map:', m)
    print('  raw labels:', sorted({r['raw'] for r in lab}))
    if not lab:
        print('  no usable labels, skipped')
        continue
    mode = MODE_OVERRIDE.get(name, 'auto')
    if name == 'counterbench':
        print('  choosing how to read the questions, using the public labels')
        tune_causal(lab)
    if any(r['pfol'].strip() for r in lab):
        # the private test set will most likely have no FOL column, so the plain english run is the one that counts
        print(f'{name} with english only (like the test set)')
        run_one(name, nl_only(lab), mode)
        print(f'\n{name} with the FOL columns given')
        run_one(name + '_fol', lab, mode)
    else:
        run_one(name, lab, mode)

TEST_LIKE = [k for k in results if not k.endswith('_fol')]

print('\nsummary')
print(f"{'dataset':16s}{'n':>8s}{'acc %':>9s}{'time s':>9s}{'ms/ex':>9s}")
for k, v in results.items():
    print(f"{k:16s}{v['n']:8d}{v['acc']:9.2f}{v['time']:9.2f}{1000 * v['time'] / max(v['n'], 1):9.2f}")
tn_ = sum(results[k]['n'] for k in TEST_LIKE)
tc_ = sum(results[k]['correct'] for k in TEST_LIKE)
tt_ = sum(results[k]['time'] for k in TEST_LIKE)
if tn_:
    OVERALL = {'n': tn_, 'acc': 100.0 * tc_ / tn_, 'time': tt_}
    print(f"{'overall':16s}{tn_:8d}{OVERALL['acc']:9.2f}{tt_:9.2f}{1000 * tt_ / tn_:9.2f}   (english-only rows, folio_fol left out)")
else:
    OVERALL = None
    print('nothing was evaluated')



ruletaker: split test, 151911 rows available, 151911 used, 151911 labelled
  column map: {'ctx': 'context', 'q': 'question', 'lab': 'label', 'id': None, 'pfol': None, 'cfol': None}
  raw labels: ['entailment', 'not entailment']
  mode: auto
  accuracy 99.97% (151870/151911), time 15.06s (parse 5.76s, reason 8.37s)
  confusion tp,tn,fp,fn = (76400, 75470, 15, 26)
  unparsed context sentences 0 of 212109, bad queries 0, fol rows 0, fol parse failures 0

counterbench: split files data_balanced_alpha_V1 1000, data_balanced_backdoor_V2 200, 1200 rows available, 1200 used, 1200 labelled
  column map: {'ctx': 'given_info', 'q': 'question', 'lab': 'answer', 'id': 'question_id', 'pfol': None, 'cfol': None}
  raw labels: ['no', 'yes']
  choosing how to read the questions, using the public labels
  causal form instead: 700 rows
    reading I1: 92.14%
    reading I2: 92.14%
  causal form plain: 250 rows
    reading P1: 96.00%
    reading P2: 96.00%
  causal form nested: 250 rows
    reading N1: 9

In [10]:
for name, wl in wrong_ex.items():
    for r, p in wl:
        print(f'[{name}] gold {r["gold"]}, pred {p}, raw label {r["raw"]}')
        print('  ctx:', r['ctx'][:400].replace('\n', ' '))
        print('  q:', r['q'][:200])

[ruletaker] gold True, pred False, raw label entailment
  ctx: The bear does not see the lion. The cow sees the lion. The lion eats the bear. The lion is round. The lion sees the cow. The rabbit eats the bear. The rabbit likes the lion. If someone sees the lion and the lion eats the cow then the cow does not see the lion. If the bear eats the rabbit then the rabbit eats the lion. If someone eats the cow then they are not round. If someone sees the bear then t
  q: The bear does not like the bear.
[ruletaker] gold False, pred True, raw label not entailment
  ctx: The bear does not see the lion. The cow sees the lion. The lion eats the bear. The lion is round. The lion sees the cow. The rabbit eats the bear. The rabbit likes the lion. If someone sees the lion and the lion eats the cow then the cow does not see the lion. If the bear eats the rabbit then the rabbit eats the lion. If someone eats the cow then they are not round. If someone sees the bear then t
  q: The lion likes the bear.


In [11]:
def compare_modes(name, n=500):
    # quick check of cwa vs owa vs auto on a slice of one dataset
    if name not in keep_rows: return print('no data for', name)
    rows = keep_rows[name][0][:n]
    for mode in ('cwa', 'owa', 'auto'):
        cache.clear()
        res, _, _ = evaluate(name, rows, mode)
        print(f"{name} {mode}: {res['acc']:.2f}% on {res['n']}")
    cache.clear()


MODE_CHECK = []
for nm in keep_rows:
    if not nm.endswith('_fol') and nm != 'counterbench':
        compare_modes(nm)


ruletaker cwa: 100.00% on 500
ruletaker owa: 73.20% on 500
ruletaker auto: 100.00% on 500
folio cwa: 63.05% on 203
folio owa: 64.04% on 203
folio auto: 64.04% on 203


## reasoning traces

In [12]:
TRACES = []


def ascii_fol(s):
    for a, b in (('∀', 'forall '), ('∃', 'exists '), ('¬', '~'), ('∧', '&'), ('∨', '|'),
                 ('⊕', ' xor '), ('→', '->'), ('↔', '<->')):
        s = s.replace(a, b)
    return s


def horn_trace(ctx, q, mode):
    horn_predict(ctx, q, mode, new_stats())
    st, deriv, given, used = cache[(ctx, mode)]
    ql = parse_query(q)
    if ql is None: return 'query not parsed'
    lines = []
    for l in ql:
        a = holds(st, l, used)
        lines.append(f'goal {fmt(l)}: {a}   (mode {used})')
        if a and l[0] and used == 'cwa':
            lines.append(f'  no proof of {fmt((False, l[1], l[2]))} (negation as failure)')
        elif a:
            explain(l, deriv, given, lines, 1, set())
        else:
            lines.append('  not derivable, closure has ' + str(sum(len(v) for v in st['p'].values())) + ' positive facts')
    return '\n'.join(lines)


def show_traces(per=2, scan=3000):
    TRACES.clear()

    def out(s):
        print(s)
        TRACES.append(s)

    for name, (rows, mode) in keep_rows.items():
        got = {True: 0, False: 0}
        out(f'\n{name} ({mode})')
        for r in rows[:scan]:
            p = predict(r, mode, new_stats())
            if got[p] >= (per if p else 1) or p != r['gold']: continue
            got[p] += 1
            out(f'--- gold {r["gold"]}, pred {p}')
            if r['pfol'].strip() and r['cfol'].strip():
                out('premises: ' + ascii_fol(r['pfol'][:600].replace('\n', ' ; ')))
                out('conclusion: ' + ascii_fol(r['cfol'][:200]))
                ok, why, proof = fol_predict(r['pfol'], r['cfol'], new_stats())
                out(why)
                out(proof)
            elif is_causal(r['ctx']):
                out('given: ' + r['ctx'][:500].replace('\n', ' '))
                out('question: ' + r['q'][:200])
                out(causal_trace(r['ctx'], r['q']))
            else:
                out('context: ' + r['ctx'][:500].replace('\n', ' '))
                out('query: ' + r['q'][:200])
                out(horn_trace(r['ctx'], r['q'], mode))
            if got[True] >= per and got[False] >= 1: break


show_traces()



ruletaker (auto)
--- gold True, pred True
context: Anne is big. Anne is cold. Anne is furry. Anne is green. Anne is red. Anne is rough. Erin is cold. Erin is furry. Erin is green. Erin is round. If something is cold then it is furry. If something is round then it is rough. If something is green and cold then it is furry. All furry things are round. Red things are big. If something is rough then it is furry. All rough things are cold. All red things are round.
query: Anne is red.
goal red(anne): True   (mode cwa)
  red(anne)   [given]
--- gold False, pred False
context: Anne is big. Anne is cold. Anne is furry. Anne is green. Anne is red. Anne is rough. Erin is cold. Erin is furry. Erin is green. Erin is round. If something is cold then it is furry. If something is round then it is rough. If something is green and cold then it is furry. All furry things are round. Red things are big. If something is rough then it is furry. All rough things are cold. All red things are round.
query: Eri

## submission

In [13]:
def run_test(path, mode):
    import pandas as pd
    df = read_table(path)
    m = colmap(list(df.columns))
    print('columns:', list(df.columns))
    print('column map:', m)
    if not m['ctx'] or not m['q']:
        raise ValueError('could not find context/query columns, edit CTX and QRY near the top of the adapter cell')
    if not m['id']: print('no id column found, using row number starting at 0')
    rows = [mkrow(rec, m, None, i) for i, rec in enumerate(df.to_dict('records'))]
    cache.clear()
    stats = new_stats()
    t0 = time.perf_counter()
    preds = [predict(r, mode, stats) for r in rows]
    tt = time.perf_counter() - t0
    out = pd.DataFrame({'id': [r['id'] for r in rows], 'label': [str(p) for p in preds]})
    out.to_csv(OUT_PATH, index=False)
    gold = [r['gold'] for r in rows]
    if all(g is not None for g in gold):
        print(f'accuracy on the test file {100.0 * sum(p == g for p, g in zip(preds, gold)) / len(rows):.2f}%')
    print(f'wrote {OUT_PATH}: {len(out)} rows, {sum(preds)} True, {len(preds) - sum(preds)} False, inference time {tt:.2f}s')
    print('unparsed sentences', int(stats['bad_sent']), 'bad queries', int(stats['bad_query']), 'fol parse failures', int(stats['fol_bad']))
    return out, tt


def public_submission():
    # no private file given: write the csv from the public held out splits (english only, same as a real test file)
    import pandas as pd
    ids, labs, n_ok, n = [], [], 0, 0
    for k in TEST_LIKE:
        rows = keep_rows[k][0]
        preds = results[k]['preds']
        for r, p in zip(rows, preds):
            ids.append(f'{k}-{r["id"]}')
            labs.append(str(p))
            n += 1
            n_ok += (p == r['gold'])
    out = pd.DataFrame({'id': ids, 'label': labs})
    out.to_csv(OUT_PATH, index=False)
    tt = sum(results[k]['time'] for k in TEST_LIKE)
    print(f'TEST_PATH not set, wrote {OUT_PATH} from the public splits: {n} rows, accuracy {100.0 * n_ok / max(n, 1):.2f}%, inference time {tt:.2f}s')
    return out, tt


if TEST_PATH:
    sub, SUB_TIME = run_test(TEST_PATH, TEST_MODE)
elif TEST_LIKE:
    sub, SUB_TIME = public_submission()
else:
    sub, SUB_TIME = None, 0.0
    print('no data and no TEST_PATH, nothing to write')


TEST_PATH not set, wrote submission.csv from the public splits: 153314 rows, accuracy 99.88%, inference time 15.35s


## readme


In [14]:
def build_readme():
    lines = [
        'README',
        '',
        'Files',
        '  reasoning_agent.ipynb   the whole system (parsers, forward chaining, resolution, causal solver, evaluation, submission)',
        '  report.pdf              short report, compiled from report.tex with pdflatex',
        '  submission.csv          columns id,label with True or False',
        '  README.txt              this file',
        '',
        'How to run',
        '  1. pip install datasets huggingface_hub pandas',
        '  2. Open the notebook and run all cells. The first run downloads RuleTaker (tasksource/ruletaker),',
        '     FOLIO (tasksource/folio) and CounterBench (read from its raw files if load_dataset fails).',
        '  3. For the hidden test file set TEST_PATH at the top (csv, tsv, json, jsonl or parquet) and run all cells again.',
        '     The file needs a context column and a query column (names like context/premises/given_info and question/conclusion are found automatically).',
        '     An id column is used if present, otherwise the row number is the id.',
        '  4. If TEST_PATH is None the submission.csv is made from the public splits instead.',
        '',
        'Settings at the top of the notebook',
        '  TEST_MODE   auto (default), cwa or owa',
        '  MAX_EVAL    cap rows per dataset for a quick run',
        '  FOL_TIMEOUT seconds per resolution proof',
        '  LOCAL_DIR   folder with files named <dataset>_<split>.csv if there is no internet',
        '',
        'Method in one paragraph',
        '  English rules and facts are parsed into literals and horn rules by regex and token rules, then solved by forward chaining',
        '  with unification and negation as failure. FOL formulas (FOLIO) are converted to clauses and solved by resolution',
        '  refutation with set of support. CounterBench stories are parsed into a small causal model and the counterfactual',
        '  question is answered by evaluating that model over the possible root assignments. No generative model is used anywhere.',
        '',
        'Limits',
        '  English that does not follow simple rule patterns (most of FOLIO) is only partly parsed, so those rows are mostly answered False.',
        '  The way CounterBench questions are read (which reading is used per question form) is chosen on the public labels.',
    ]
    with open(README_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    print('wrote', README_PATH)


build_readme()
print('\nfinal numbers')
if OVERALL:
    print(f"accuracy {OVERALL['acc']:.2f}% on {OVERALL['n']} english rows, inference time {OVERALL['time']:.2f}s")
    for k in TEST_LIKE:
        print(f"  {k}: {results[k]['acc']:.2f}% on {results[k]['n']} rows, {results[k]['time']:.2f}s")
print(f'submission inference time {SUB_TIME:.2f}s')


wrote README.txt

final numbers
accuracy 99.88% on 153314 english rows, inference time 15.35s
  ruletaker: 99.97% on 151911 rows, 15.06s
  counterbench: 93.75% on 1200 rows, 0.25s
  folio: 64.04% on 203 rows, 0.03s
submission inference time 15.35s
